In [71]:
!pip install nltk

import nltk
nltk.download('wordnet')
nltk.download('stopwords')


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[nltk_data] Downloading package wordnet to /Users/riki/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/riki/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [72]:
import pandas as pd

# 1. Load local datasets directly
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

# 2. Merge both splits for Pure EDA
df_raw = pd.concat([train_df, test_df], ignore_index=True)

# 3. Check the results
print(f"Total combined raw data: {len(df_raw)} rows")
print("\nTop 3 rows preview:")
print(df_raw.head(3))



Total combined raw data: 13083 rows

Top 3 rows preview:
                                                           text      category
0                                I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived after 2 weeks?  card_arrival
2    I have been waiting over a week. Is the card still coming?  card_arrival


In [73]:
df_raw['text'].sample(25, random_state=411).tolist()

# Based from sample Symbol , rare characters, bloat are almost to none shown

['Is there a charge for a foreign currency exchange?',
 'Could you please tell me your exchange rates?',
 'I would like to know how exchange rates are calculated when transferring funds.',
 'What ATMs can I use my card at?',
 "I can't get my Google Pay Top to work. Are there any issues on your end?",
 'How much does it cost for a physical card?',
 'Why is there an extra dollar charged to my account?',
 'I will not be able to verify my identity.',
 "Can I get a card despite the fact I'm not in the UK?",
 'Why have I not yet received my virtual card?',
 "I think something went wrong with my card delivery as I haven't received it yet.",
 'What should I do to top up using my Apple Watch?',
 'Why does my payment appear as pending?',
 'How long will it take to get my refund',
 'My account shows the card payment cancelled.',
 'how can i transfer money from american express card',
 'I just had a look at my statement. Why have I been changed for using the ATM?',
 'I have a canceled top-up and w

In [74]:
# 1. Set Pandas display option to show full text (prevent truncation)
pd.set_option('display.max_colwidth', None)

# 2. Check the top 3 rows again (now fully visible)
print("=== FULL TEXT PREVIEW ===")
print(df_raw.head(3))

# 3. PURE NLP EDA: Word Count & Prefix Analysis
print("\n=== TEXT ANATOMY & PREFIX CLUSTERING ===")

# Calculate word count per text
df_raw['text'] = df_raw['text'].astype('string')

df_raw['word_count'] = df_raw['text'].str.split().str.len()
df_raw['char_count'] = df_raw['text'].str.len()

print("Word Count Statistics:")
print(df_raw['word_count'].describe(percentiles=[.01,.05,.5,.9,.95,.99]).round(2))

print("\nChar Count Statistics:")
print(df_raw['char_count'].describe(percentiles=[.01,.05,.5,.9,.95,.99]).round(2))

print("\nNull word_count:", df_raw['word_count'].isna().sum())
print("Null char_count:", df_raw['char_count'].isna().sum())

print("\nBlank or whitespace check:")
print((df_raw['word_count'] == 0).sum())

blank_mask = df_raw['text'].fillna('').str.strip().eq('')
blank_mask.sum()


# Extract the prefix from the 'category' column (e.g., 'card_arrival' -> 'card')
df_raw['prefix'] = df_raw['category'].apply(lambda x: str(x).split('_')[0])
print("\nTop 5 Prefix Clusters (Highest Confusion Potential):")
print(df_raw['prefix'].value_counts().head(5))

=== FULL TEXT PREVIEW ===
                                                           text      category
0                                I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived after 2 weeks?  card_arrival
2    I have been waiting over a week. Is the card still coming?  card_arrival

=== TEXT ANATOMY & PREFIX CLUSTERING ===
Word Count Statistics:
count    13083.00
mean        11.71
std          7.64
min          2.00
1%           4.00
5%           5.00
50%         10.00
90%         21.00
95%         28.00
99%         42.00
max         79.00
Name: word_count, dtype: float64

Char Count Statistics:
count    13083.0
mean       58.24
std        39.56
min         13.0
1%          20.0
5%          25.0
50%         46.0
90%        104.0
95%        145.0
99%        216.0
max        433.0
Name: char_count, dtype: Float64

Null word_count: 0
Null char_count: 0

Blank or whitespace check:
0

Top 5 Prefix Clusters (Highest Confusion Potential):

In [75]:
# ============================================================
# Text Length Analysis
# ============================================================

# 1. Get the 20 longest texts based on character count
longest_idx = df_raw['char_count'].nlargest(20).index
longest_texts = df_raw.loc[
    longest_idx,
    ['text', 'word_count', 'char_count']
].sort_values('char_count', ascending=False)


# 2. Get the 20 shortest texts based on word count
shortest_idx = df_raw['word_count'].nsmallest(20).index
shortest_texts = df_raw.loc[
    shortest_idx,
    ['text', 'word_count', 'char_count']
].sort_values('word_count', ascending=True)


# 3. Calculate P99 thresholds
p99_word = df_raw['word_count'].quantile(0.99)
p99_char = df_raw['char_count'].quantile(0.99)


# 4. Identify outliers above P99
outlier_word = df_raw[df_raw['word_count'] > p99_word]
outlier_char = df_raw[df_raw['char_count'] > p99_char]


# ============================================================
# Print Results
# ============================================================

print("=" * 70)
print("TEXT LENGTH EDA")
print("=" * 70)

print("\n[1] 20 LONGEST TEXTS BY CHARACTER COUNT")
print("-" * 70)
print(
    longest_texts.to_string(index=False)
)

print("\n[2] 20 SHORTEST TEXTS BY WORD COUNT")
print("-" * 70)
print(
    shortest_texts.to_string(index=False)
)

print("\n[3] P99 THRESHOLDS")
print("-" * 70)
print(f"P99 word count : {p99_word:.2f}")
print(f"P99 char count : {p99_char:.2f}")

print("\n[4] OUTLIER COUNTS (> P99)")
print("-" * 70)
print(f"Word count outliers : {len(outlier_word)}")
print(f"Char count outliers : {len(outlier_char)}")

print("\n" + "=" * 70)

TEXT LENGTH EDA

[1] 20 LONGEST TEXTS BY CHARACTER COUNT
----------------------------------------------------------------------
                                                                                                                                                                                                                                                                                                                                                                                                                                             text  word_count  char_count
Hearing back from us regarding your important verification results may take 10 minutes to one hour time.  If verification results do fail, double-check to make sure all of your images are clear --  make sure your photos have no glare or blurring. Note: These photos need to be readable as well.  You also need to be 18 years of age or older.  You must be a resident of Switzerland or the European Economic Area to 

# Summary for Text Anatomy and Prefix Clustering
1. Based on Word count statistic found outlier for max count above 75%, need to check the words.

2. Based on the prefix extraction, topics related to 'Card' (1,687 rows), 'Top Up' (967 rows), and 'Transfer' (744 rows) are the largest contributors to Class Overlap. This is predicted to be the primary blind spot for the TF-IDF model due to the high lexical similarity between different intents.

3. Shortest text list doesn't show any issue with typo or noise, everything still intent focused on banking


In [76]:
# Weird character token, bloat,mojibake and symbol check


import collections
import pandas as pd

# 1. Sample text and safe for NaN
sample_text = "".join(df_raw["text"].dropna().astype(str).head(5000))

# 2. Count characters and filter weird characters
char_counts = collections.Counter(sample_text)

weird_chars = {
    char: count
    for char, count in char_counts.items()
    if not char.isalnum() and not char.isspace()
}

# 3. Sort by frequency
sorted_weird_chars = sorted(
    weird_chars.items(),
    key=lambda item: item[1],
    reverse=True
)

# 4. Convert to DataFrame for clean display
df_weird_chars = pd.DataFrame(
    sorted_weird_chars,
    columns=["character", "count"]
)

# 5. Show raw representation to reveal hidden characters like \n, \t, \r
df_weird_chars["raw_form"] = df_weird_chars["character"].apply(repr)

# 6. Print top 25 weird characters
print("=== TOP 25 WEIRD CHARACTERS ===")
display(df_weird_chars.head(25))

=== TOP 25 WEIRD CHARACTERS ===


,character,count,raw_form
0,?,3217,'?'
1,.,2079,'.'
2,',1357,"""'"""
3,",",705,"','"
4,-,297,'-'
5,!,205,'!'
6,$,83,'$'
7,£,24,'£'
8,€,21,'€'
9,(,7,'('


# Summary for mojibake and weird characters


✅ Normal punctuation found → clean/remove in cleaning engineering phase
✅ Apostrophe found → expand contractions in cleaning phase 
   (NOTE: skip expansion if using LLM/modern transformer)
✅ Currency symbols found ($, £, €) → mask as [MONEY] in domain cleaning phase
✅ Mojibake/HTML symbols not found → skip encoding normalization

In [77]:
# Extract the exact row with the maximum word count
max_length = df_raw['word_count'].max()
outlier_data = df_raw[df_raw['word_count'] == max_length]

# Display the outlier text and its intent category
print(f"=== THE {max_length}-WORD OUTLIER ===")
print(f"Target Intent: {outlier_data['category'].values[0]}")
print(f"Full Text:\n{outlier_data['text'].values[0]}")

=== THE 79-WORD OUTLIER ===
Target Intent: unable_to_verify_identity
Full Text:
Hearing back from us regarding your important verification results may take 10 minutes to one hour time.  If verification results do fail, double-check to make sure all of your images are clear --  make sure your photos have no glare or blurring. Note: These photos need to be readable as well.  You also need to be 18 years of age or older.  You must be a resident of Switzerland or the European Economic Area to open a new account.


In [78]:
# Check target distribution for potential class imbalance
intent_counts = df_raw['category'].value_counts()

print("=== CLASS DISTRIBUTION ===")
print(f"Total unique intents: {len(intent_counts)} classes")

print("\nTop 5 Most Frequent Intents:")
print(intent_counts.head(5))

print("\nBottom 5 Least Frequent Intents:")
print(intent_counts.tail(5))

=== CLASS DISTRIBUTION ===
Total unique intents: 77 classes

Top 5 Most Frequent Intents:
category
card_payment_fee_charged                            227
direct_debit_payment_not_recognised                 222
balance_not_updated_after_cheque_or_cash_deposit    221
wrong_amount_of_cash_received                       220
cash_withdrawal_charge                              217
Name: count, dtype: int64

Bottom 5 Least Frequent Intents:
category
lost_or_stolen_card         122
card_swallowed              101
card_acceptance              99
virtual_card_not_working     81
contactless_not_working      75
Name: count, dtype: int64


# summary for class distribution
The dataset has a moderate class imbalance (max 227 vs min 75 samples). While not extreme, it requires attention during modeling to ensure minority intents (like `contactless_not_working`) are not ignored by the algorithm.


# 🚀 Actionable Next Steps for Modeling
1. **Balance the Data (Optional):** Use **Data Augmentation** for bottom classes, or **Undersample** top classes if the model heavily favors the majority.
2. **Apply Class Weights (Recommended):** Configure the loss function to penalize the model more for misclassifying rare intents. (Often an easier alternative to modifying the actual data).
3. **Use Robust Metrics (Mandatory 🚨):** **DO NOT rely on standard Accuracy.** must use **Macro F1-Score** and inspect the **Confusion Matrix** to get a truthful evaluation of the model's performance across all classes.
4. **Design Fallbacks (Production Only):** For critical but rare intents, set strict confidence

In [79]:
print("=== SANITY CHECK: MISSING & EMPTY TEXT ===")
# Check for explicit NULL/NaN values
null_count = df_raw['text'].isnull().sum()

# Check for empty strings or whitespace-only strings
# fillna('') for nan replacement
empty_count = (
    df_raw['text']
    .fillna('')
    .str.strip()
    .eq('')
    .sum()
)

print(f"Null values count: {null_count}")
print(f"Empty strings count: {empty_count}")

print("\n=== SANITY CHECK: DUPLICATE TEXT ===")
# Check for exact duplicate text across the entire dataset
duplicate_count = df_raw.duplicated(subset=['text']).sum()
print(f"Total duplicate rows: {duplicate_count}")

=== SANITY CHECK: MISSING & EMPTY TEXT ===
Null values count: 0
Empty strings count: 0

=== SANITY CHECK: DUPLICATE TEXT ===
Total duplicate rows: 0


# summary for sanity check

The dataset is exceptionally clean "out-of-the-box". Found `0` null values, `0` empty strings, and `0` duplicate rows.


In [80]:
universal_pii_patterns = {
    "email_generic": r'\b[\w.-]+@[\w.-]+\.\w{2,}\b',
    "credit_card_universal": r'\b(?:\d[ -]?){13,19}\b',  # 13-19 digits (Visa, MC, Amex)
    "phone_generic": r'\b(?:\+?\d{1,3}[\s.-]?)?\(?\d{2,4}\)?[\s.-]?\d{3,4}[\s.-]?\d{3,4}\b',
    "dummy_numbers": r'\b\d{6,}\b'  
}

print("=== UNIVERSAL PII / DUMMY SCAN ===")
for name, pattern in universal_pii_patterns.items():
    count = df_raw['text'].str.contains(pattern, regex=True, case=False, na=False).sum()
    print(f"{name:25s}: {count}")

=== UNIVERSAL PII / DUMMY SCAN ===
email_generic            : 0
credit_card_universal    : 0
phone_generic            : 0
dummy_numbers            : 0


# Summary for PII Privacy and Security data

The dataset already scrubed and cleaned as no PII found , PII masking step can be skipped for this dataset.

In [81]:
import re
from collections import Counter

S_OK = {'it','that','there','here','what','where','who','how','why','when','he','she'}

toks = re.findall(r"\b[a-z]+'[a-z]+\b", " ".join(df_raw['text'].str.lower()))
clean = [t for t in toks if not (t.endswith("'s") and t.split("'")[0] not in S_OK)]

Counter(clean).most_common(20)

[("didn't", 395),
 ("don't", 379),
 ("i'm", 350),
 ("can't", 279),
 ("it's", 271),
 ("hasn't", 222),
 ("isn't", 209),
 ("i've", 202),
 ("what's", 200),
 ("doesn't", 116),
 ("haven't", 115),
 ("won't", 99),
 ("wasn't", 73),
 ("there's", 62),
 ("i'd", 51),
 ("shouldn't", 32),
 ("wouldn't", 32),
 ("couldn't", 31),
 ("that's", 25),
 ("aren't", 11)]

# Summary for contraction
60% of the top contractions are negation or rejection, so the negation words must survive cleaning. The corpus uses formal tone almost no slang, so no slang need processing. Most utterances are personal and question based , which marks the register as personal complaint — these first-person and wh- contractions are style markers, not intent discriminators; the real
discriminators remain rare domain words.

In [82]:
print("=== HYGIENE CHECK: CROSS-SPLIT LEAKAGE ===")

# Convert text columns to Set for blazing fast intersection comparison
train_texts = set(train_df['text'])
test_texts = set(test_df['text'])

# Find overlaps between Train and Test sets
data_leakage = train_texts.intersection(test_texts)

print(f"Total overlapping texts (Train & Test): {len(data_leakage)}")

if len(data_leakage) > 0:
    print("WARNING: Data leakage detected! Investigation required.")
else:
    print("SAFE: No cross-split leakage found.")

=== HYGIENE CHECK: CROSS-SPLIT LEAKAGE ===
Total overlapping texts (Train & Test): 0
SAFE: No cross-split leakage found.


# summary for Hygiene check

The dataset does not have any overlapping data that could cause leakage between train and test, thus is safe to use for train and test

# Cleaning Engineering

In [83]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
NEGATION = {'not', 'no', 'nor', 'never', 'without'}
STOPWORDS = set(stopwords.words('english')) - NEGATION

CONTRACTIONS = {
    r"can't": "can not", r"won't": "will not", r"n't": " not",
    r"'re": " are", r"'s": " is", r"'d": " would",
    r"'ll": " will", r"'ve": " have", r"'m": " am",
}

# ─── FASE 1: Universal Basics ───
def phase_universal(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    for pat, repl in CONTRACTIONS.items():
        text = re.sub(pat, repl, text)
    return text

# ─── FASE 2: Domain-Specific (sebelum noise!) ───
def phase_domain(text):
    text = re.sub(r'\*{4}\s?\d{4}', ' cardnumber', text)
    text = re.sub(r'\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b', ' cardnumber', text)
    text = re.sub(r'\b\d{8,17}\b', ' accountnumber', text)
    text = re.sub(r'[$£€¥]\s?\d[\d,]*\.?\d*', ' moneyamount', text)
    text = re.sub(r'\b\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4}\b', ' date', text)
    return text

# ─── FASE 3: Noise Removal ───
def phase_noise(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ─── FASE 4: NLP Formal ───
def phase_nlp(text):
    tokens = text.split()
    return " ".join(
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in STOPWORDS and len(t) > 1
    )

# ─── ORCHESTRATOR ───
def clean_text(text, verbose=False):
    if not text or not isinstance(text, str):
        return ""
    steps = [
        ("FASE 1 Universal", phase_universal),
        ("FASE 2 Domain",    phase_domain),
        ("FASE 3 Noise",     phase_noise),
        ("FASE 4 NLP",       phase_nlp),
    ]
    result = text
    for name, func in steps:
        result = func(result)
        if verbose:
            print(f"{name:>16} → {result}")
    return result

In [84]:
df_raw['clean_text'] = df_raw['text'].apply(clean_text)

# Jaccard Similarity analysis

In [85]:
# Verification before Jaccard

# ─── CLEANING VERIFICATION (pre-Jaccard) ───
# 1. Before/After visual check
for i in df_raw.sample(5, random_state=42).index:
    print("RAW  :", df_raw.loc[i, 'text'])
    print("CLEAN:", df_raw.loc[i, 'clean_text'])
    print("-" * 60)

# 2. Check for empty rows after cleaning
empty_clean = (df_raw['clean_text'].str.strip() == '').sum()
print(f"Empty rows after cleaning: {empty_clean}")

# 3. Clean token distribution
df_raw['clean_tokens'] = df_raw['clean_text'].str.split().str.len().fillna(0)
print(df_raw['clean_tokens'].describe())
print(f"Rows with <= 1 token: {(df_raw['clean_tokens'] <= 1).sum()}")

RAW  : Please explain the exchange rates.
CLEAN: please explain exchange rate
------------------------------------------------------------
RAW  : I got some cash of an ATM earlier but this shows up as pending in the app. How can this be still pending, I've got the cash already?
CLEAN: got cash atm earlier show pending app still pending got cash already
------------------------------------------------------------
RAW  : I just replaced my phone but how do I use my card information?
CLEAN: replaced phone use card information
------------------------------------------------------------
RAW  : I wasn't able to do a transfer to an account
CLEAN: not able transfer account
------------------------------------------------------------
RAW  : I don't recognize a charge on my statement.
CLEAN: not recognize charge statement
------------------------------------------------------------
Empty rows after cleaning: 0
count    13083.000000
mean         5.336161
std          3.645564
min          1.0000

In [91]:
def show_few_token_rows(df, max_tokens=1):
    """Print rows with <= max_tokens tokens after cleaning."""
    short = df[df['clean_tokens'] <= max_tokens]
    print(f"Found {len(short)} rows with <= {max_tokens} token:\n")
    for idx, row in short.iterrows():
        print(f"[{idx:>5}] {row['category']}")
        print(f"        RAW  : {row['text']}")
        print(f"        CLEAN: '{row['clean_text']}'")
        print("-" * 70)

In [92]:
show_few_token_rows(df_raw)

Found 48 rows with <= 1 token:

[   87] card_arrival
        RAW  : Where is my card?
        CLEAN: 'card'
----------------------------------------------------------------------
[  368] exchange_rate
        RAW  : How do you do exchanges?
        CLEAN: 'exchange'
----------------------------------------------------------------------
[  660] extra_charge_on_statement
        RAW  : Why was I charged more than I should have been?
        CLEAN: 'charged'
----------------------------------------------------------------------
[  886] fiat_currency_support
        RAW  : What currencies can I have?
        CLEAN: 'currency'
----------------------------------------------------------------------
[ 1020] card_delivery_estimate
        RAW  : When will my card be here?
        CLEAN: 'card'
----------------------------------------------------------------------
[ 1172] automatic_top_up
        RAW  : Can I auto-top up?  How do I do it?
        CLEAN: 'autotop'
--------------------------------

In [88]:
# ─── PRE-JACCARD FILTER: EXCLUDE SHORT UTTERANCES ───
# We only want Jaccard to compare texts with >= 2 tokens to avoid 
# false 100% matches caused by single-word stopword-heavy questions.

df_jaccard = df_raw[df_raw['clean_tokens'] > 1].copy()

print(f"Total rows in dataset   : {len(df_raw)}")
print(f"Rows excluded (<=1 tok) : {len(df_raw) - len(df_jaccard)}")
print(f"Rows ready for Jaccard  : {len(df_jaccard)}")
print(f"\n✅ Dataset is now clean and ready for Jaccard Similarity!")

Total rows in dataset   : 13083
Rows excluded (<=1 tok) : 48
Rows ready for Jaccard  : 13035

✅ Dataset is now clean and ready for Jaccard Similarity!


In [94]:
from itertools import combinations
from collections import Counter

def jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return 0.0
    return len(set_a & set_b) / len(union)

def build_intent_vocab(df, text_col='clean_text'):
    vocab = {}
    for intent, group in df.groupby('category'):
        tokens = set()
        for text in group[text_col]:
            tokens.update(text.split())
        vocab[intent] = tokens
    return vocab

def build_word_df(vocab):
    word_df = Counter()
    for tokens in vocab.values():
        word_df.update(tokens)
    return word_df

def build_review_queue(df, text_col='clean_text', min_score=0.30,
                       df_max=4, top_n=15):
    vocab   = build_intent_vocab(df, text_col)
    word_df = build_word_df(vocab)
    queue = []
    for intent_a, intent_b in combinations(sorted(vocab), 2):
        words_a, words_b = vocab[intent_a], vocab[intent_b]
        score = jaccard(words_a, words_b)
        if score < min_score:
            continue
        shared        = words_a & words_b
        shared_low_df = sorted(w for w in shared if word_df[w] <= df_max)
        queue.append({
            'intent_a':      intent_a,
            'intent_b':      intent_b,
            'score':         round(score, 3),
            'n_shared':      len(shared),
            'shared_low_df': shared_low_df,
        })
    queue.sort(key=lambda row: row['score'], reverse=True)
    return queue[:top_n], word_df

def print_review_queue(queue):
    for rank, row in enumerate(queue, 1):
        print(f"[{rank:>2}] jaccard={row['score']:.3f} | shared={row['n_shared']:>3} words")
        print(f"     {row['intent_a']}")
        print(f"     <-> {row['intent_b']}")
        low_df = row['shared_low_df']
        print(f"     LOW-DF SHARED (high-IDF): {low_df if low_df else '-'}")
        print("-" * 70)

In [96]:
import json

queue_all, word_df = build_review_queue(df_jaccard, min_score=0.20, top_n=100000)
print(f"Total pairs >= 0.20: {len(queue_all)}")

band_low  = [q for q in queue_all if q['score'] < 0.30]
band_high = [q for q in queue_all if q['score'] >= 0.30]
low_empty  = sum(1 for q in band_low  if not q['shared_low_df'])
high_empty = sum(1 for q in band_high if not q['shared_low_df'])
print(f"Band 0.20-0.30: {len(band_low):>3} pairs | common-only (high-DF): {low_empty}")
print(f"Band >= 0.30  : {len(band_high):>3} pairs | common-only (high-DF): {high_empty}")

review_top = band_high[:20]
leak_candidates = [q for q in queue_all if q['shared_low_df']]
print(f"True-leak candidates (any score): {len(leak_candidates)}")

with open('jaccard_queue_full.json', 'w') as f:
    json.dump(queue_all, f, indent=2)

print_review_queue(review_top)
print("\n=== TRUE-LEAK CANDIDATES ===")
print_review_queue(leak_candidates[:15])

Total pairs >= 0.20: 379
Band 0.20-0.30: 354 pairs | common-only (high-DF): 123
Band >= 0.30  :  25 pairs | common-only (high-DF): 3
True-leak candidates (any score): 253
[ 1] jaccard=0.361 | shared=104 words
     pending_card_payment
     <-> pending_transfer
     LOW-DF SHARED (high-IDF): -
----------------------------------------------------------------------
[ 2] jaccard=0.358 | shared=102 words
     balance_not_updated_after_bank_transfer
     <-> balance_not_updated_after_cheque_or_cash_deposit
     LOW-DF SHARED (high-IDF): ['banking', 'stayed', 'updating']
----------------------------------------------------------------------
[ 3] jaccard=0.351 | shared=111 words
     card_payment_not_recognised
     <-> direct_debit_payment_not_recognised
     LOW-DF SHARED (high-IDF): ['approve', 'authorize', 'fraudulent', 'odd', 'recognise', 'recognizing', 'record', 'suspicious', 'unfamiliar', 'unknown', 'unusual']
----------------------------------------------------------------------
[ 4] j

# Summary of Jaccard similarity analysis

**Method** : Pairwise Jaccard on cleaned dataset with intent vocabularies , The token-length threshold was lowered from 2 to 1  because there are countries initial such as UK, US and to make sure they survived. Trying to find rare-word spotlight (word_df <= 4) that separates true discriminator leaks from genre overlap.


**Results:**
- Pairs >= 0.20: 379 total
- Band >= 0.30: 25 pairs | 88% true leaks (22/25) → Tier 1
- Band 0.20-0.30: 354 pairs | 65% true leaks (231/354) → Tier 2 (archive)
- True-leak candidates overall: 253

**Threshold = 0.30 (evidence-based):** the share of pairs that actually
share a rare discriminator word (true leaks) jumps from 65% in the
0.20-0.30 band to 88% in the >= 0.30 band. Keeping only pairs >= 0.30
makes the manual review list small and mostly relevant. Below 0.30, most
pairs look similar only because of common high-DF words (card, pending,
transfer), not because of real semantic overlap - so they are archived
(Tier 2) for later error analysis.

**Top confusable pairs (Tier 1, curated from 25 pairs >= 0.30):**
- `pending_card_payment` ↔ `pending_transfer`
  (common-term overlap, no rare leak — safe pair, validating the threshold)
- `card_payment_not_recognised` ↔ `direct_debit_payment_not_recognised`
  (heavy discriminator leak: recognise, authorize, fraudulent, suspicious, unfamiliar, unusual)
- `top_up_by_bank_transfer_charge` ↔ `top_up_by_card_charge`
  (channel-based leak: incur, refill)
- `reverted_card_payment?` ↔ `top_up_reverted`
  (action-based leak: revert, reverted, deducted)

*The full Tier 1 list (25 pairs) and all Tier 2 pairs are archived in `jaccard_queue_full.json` for post-mortem analysis.*

**Cross-validation:** The pairs flagged by Jaccard (pending_card_payment ↔ 
pending_transfer, card_payment_not_recognised ↔ direct_debit_not_recognised, 
etc.) belong to the same prefix clusters identified in the anatomy EDA 
(card_*, top_*, pending_*, transfer_*, declined_*). Two independent methods 
converge on the same TF-IDF blind spots — strong evidence these are real 
confusion risks, not method artifacts.


**Action plan (tiered triage):**

Not every confusable pair deserves the same response, so the queue is split
by confidence and each tier gets a different treatment:

1. **Tier 1 - Proactive fix (25 pairs, score >= 0.30).**
   These are high-confidence confusions, so we act on them *before* modeling.
   The TF-IDF vectorizer uses `ngram_range=(1,2)` so bigrams create
   intent-specific features (e.g., "pending card" vs "pending transfer") and
   break the shared-discriminator ties found above. The classifier uses
   `class_weight='balanced'` to protect minority intents, and the model is
   judged with Macro-F1 (not accuracy) so small intents weigh as much as big
   ones.

2. **Tier 2 - Reactive watchlist (354 pairs, score 0.20-0.30; 231 of them share at least one low-DF discriminator).**
   These are medium-confidence confusions: real signal, but too noisy to fix
   blindly. Instead of burning review time on 354 pairs now, the full queue is
   archived in `jaccard_queue_full.json`. After training, this file is opened
   *only* when the confusion matrix shows a specific pair misbehaving - a
   cheap, targeted post-mortem instead of a blind mass review.

In short: fix the certain problems now, and keep the uncertain ones on file
until the model itself points at them.